# 🎯 Defense-AI — GRPO Fine-Tuning on Google Colab

## ▶️ How to Run

**Step 1:** Click `Runtime` → `Change runtime type` → select **T4 GPU** → Save

**Step 2:** Click `Runtime` → `Run all` (or press `Ctrl+F9`)

That's it! The notebook runs fully automatically:
- Installs dependencies (~5 min)
- Runs GRPO training (~45 min)
- Evaluates before vs after
- Uploads trained model to HuggingFace Hub

---

**Links:**
- 🤗 Space: https://huggingface.co/spaces/Bhaskar111/defense-ai
- 📦 Radar Adapter: https://huggingface.co/Bhaskar111/defense-rl-radar-adapter
- 📦 Actor Adapter: https://huggingface.co/Bhaskar111/defense-rl-actor-adapter

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Check GPU
# ═══════════════════════════════════════════════════════════════
!nvidia-smi
import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('❌ No GPU found! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Clone repo
# ═══════════════════════════════════════════════════════════════
import os
if not os.path.exists('defense-ai'):
    !git clone https://huggingface.co/spaces/Bhaskar111/defense-ai defense-ai
    print('✅ Cloned!')
else:
    %cd defense-ai
    !git pull
    print('✅ Updated!')
    %cd ..

%cd defense-ai
!ls

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Install dependencies (~5 min)
# ═══════════════════════════════════════════════════════════════
!pip install -q \
    transformers>=4.45.0 \
    trl>=0.11.0 \
    peft>=0.12.0 \
    accelerate>=0.30.0 \
    datasets>=2.20.0 \
    bitsandbytes>=0.43.0 \
    huggingface_hub \
    fastapi pydantic>=2.0.0 httpx openai

print('\n✅ All dependencies installed!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Set HuggingFace Token
# ═══════════════════════════════════════════════════════════════
# HOW TO ADD TOKEN:
#   1. Click the 🔑 key icon in the left sidebar
#   2. Click 'Add new secret'
#   3. Name: HF_TOKEN   Value: hf_your_token_here
#   4. Toggle 'Notebook access' ON

import os
from huggingface_hub import whoami

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print(f'✅ Token loaded from Colab Secrets')
except Exception:
    HF_TOKEN = 'hf_YOUR_TOKEN_HERE'  # ← paste your token here if not using Secrets
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('⚠️  Using manually pasted token')

try:
    info = whoami(token=HF_TOKEN)
    print(f'✅ Logged in as: {info["name"]}')
except Exception as e:
    print(f'❌ Token error: {e}\n   Get a token at: https://huggingface.co/settings/tokens')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Evaluate BEFORE training (baseline)
# ═══════════════════════════════════════════════════════════════
# This runs 30 episodes with the BASE Qwen model (no LoRA)
# Saves scores to compare against after training
# Expected time: ~8 minutes

import json, re

print('=' * 55)
print('  BASELINE EVALUATION (before GRPO training)')
print('=' * 55)

!python train.py \
    --backend local \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --episodes 30 \
    --epochs 1 \
    --dry-run \
    --device cuda \
    2>&1 | tee /tmp/baseline_eval.txt

# Parse scores from output
baseline_scores = {}
with open('/tmp/baseline_eval.txt') as f:
    for line in f:
        m = re.search(r'task_(easy|medium|hard).*score=([0-9.]+)', line)
        if m:
            task, score = m.group(1), float(m.group(2))
            baseline_scores.setdefault(task, []).append(score)

print('\n--- BASELINE SCORES ---')
for task, scores in baseline_scores.items():
    avg = sum(scores)/len(scores)
    solved = sum(1 for s in scores if s >= 0.80)
    print(f'  task_{task}: avg={avg:.4f}  solved={solved}/{len(scores)}')

# Save for later comparison
with open('/tmp/baseline.json', 'w') as f:
    json.dump({k: sum(v)/len(v) for k,v in baseline_scores.items()}, f)
print('\n✅ Baseline saved!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — GRPO Training (~45 min on T4)
# ═══════════════════════════════════════════════════════════════
# What happens:
#   1. Collects 100 episodes (task_easy + task_medium + task_hard)
#   2. Builds GRPO dataset (436 train / 78 eval samples)
#   3. Fine-tunes Agent 1 (Radar) with LoRA for 3 epochs
#   4. Fine-tunes Agent 2 (Actor) with LoRA for 3 epochs
#   5. Saves adapters to ./checkpoints/

print('=' * 55)
print('  GRPO TRAINING — Qwen/Qwen2.5-1.5B-Instruct')
print('=' * 55)
print('  Estimated time: 40-60 minutes on T4 GPU')
print('=' * 55)

!python train.py \
    --backend local \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --episodes 100 \
    --epochs 3 \
    --batch-size 1 \
    --lora-rank 16 \
    --device cuda \
    --checkpoint-dir checkpoints

print('\n✅ GRPO training complete!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — Evaluate AFTER training
# ═══════════════════════════════════════════════════════════════
# Loads LoRA adapters from checkpoints and runs 30 evaluation episodes

import json, re

print('=' * 55)
print('  POST-TRAINING EVALUATION')
print('=' * 55)

!python train.py \
    --backend local \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --episodes 30 \
    --epochs 1 \
    --dry-run \
    --resume checkpoints \
    --device cuda \
    2>&1 | tee /tmp/after_eval.txt

after_scores = {}
with open('/tmp/after_eval.txt') as f:
    for line in f:
        m = re.search(r'task_(easy|medium|hard).*score=([0-9.]+)', line)
        if m:
            task, score = m.group(1), float(m.group(2))
            after_scores.setdefault(task, []).append(score)

print('\n✅ Evaluation complete!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — BEFORE vs AFTER Comparison Report
# ═══════════════════════════════════════════════════════════════

import json

# Load baseline
with open('/tmp/baseline.json') as f:
    baseline = json.load(f)

after = {k: sum(v)/len(v) for k,v in after_scores.items()}

print()
print('╔══════════════════════════════════════════════════════════╗')
print('║          DEFENSE-AI — GRPO TRAINING RESULTS              ║')
print('╠══════════════╦═════════════╦═════════════╦═══════════════╣')
print('║ Task         ║ Before GRPO ║ After GRPO  ║ Improvement   ║')
print('╠══════════════╬═════════════╬═════════════╬═══════════════╣')

total_before, total_after = 0, 0
for task in ['easy', 'medium', 'hard']:
    b = baseline.get(task, 0)
    a = after.get(task, 0)
    delta = a - b
    arrow = '▲' if delta > 0 else '▼'
    total_before += b
    total_after  += a
    print(f'║ task_{task:<7} ║    {b:.4f}    ║    {a:.4f}    ║ {arrow} {delta:+.4f}      ║')

print('╠══════════════╬═════════════╬═════════════╬═══════════════╣')
mean_b = total_before / 3
mean_a = total_after / 3
delta_m = mean_a - mean_b
print(f'║ MEAN         ║    {mean_b:.4f}    ║    {mean_a:.4f}    ║ ▲ {delta_m:+.4f}      ║')
print('╚══════════════╩═════════════╩═════════════╩═══════════════╝')

print()
if mean_a > mean_b:
    pct = (mean_a - mean_b) / mean_b * 100
    print(f'✅ Model improved by {pct:.1f}% after GRPO training!')
else:
    print('ℹ️  Try increasing episodes or epochs for better results.')

# Solve rate
print()
print('Solve rate (score ≥ 0.80):')
for task in ['easy', 'medium', 'hard']:
    b_scores = baseline_scores.get(task, [])
    a_scores = after_scores.get(task, [])
    b_rate = sum(1 for s in b_scores if s >= 0.80) / max(len(b_scores),1) * 100
    a_rate = sum(1 for s in a_scores if s >= 0.80) / max(len(a_scores),1) * 100
    print(f'  task_{task}: {b_rate:.0f}% → {a_rate:.0f}%')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9 — Upload trained adapters to HuggingFace Hub
# ═══════════════════════════════════════════════════════════════

from huggingface_hub import HfApi, create_repo
import os

api      = HfApi(token=os.environ['HF_TOKEN'])
USERNAME = 'Bhaskar111'  # ← your HuggingFace username

for agent in ['radar', 'actor']:
    path    = f'checkpoints/{agent}'
    repo_id = f'{USERNAME}/defense-rl-{agent}-adapter'

    if os.path.exists(path) and os.listdir(path):
        print(f'Uploading {agent} adapter...')
        create_repo(repo_id, repo_type='model', exist_ok=True,
                    token=os.environ['HF_TOKEN'])
        api.upload_folder(
            folder_path = path,
            repo_id     = repo_id,
            repo_type   = 'model',
            token       = os.environ['HF_TOKEN'],
        )
        print(f'✅ Saved → https://huggingface.co/{repo_id}')
    else:
        print(f'❌ No checkpoint at {path} — run Cell 6 first')

print('\n🎉 Done! Your trained models are on HuggingFace Hub.')